In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Classification Exercise

We build a binary classifier on the California Schools dataset to predict whether a district's
average test score is **above or below the median**.

Steps:
1. Load the data and create a binary target.
2. Split into training and test sets.
3. Fit Linear Discriminant Analysis (LDA) and Logistic Regression; compare accuracy.
4. Visualise the decision boundary for a pair of features.

In [ ]:
df = pd.read_csv('../datasets/Caschool.csv', sep=';')
print('Shape:', df.shape)
display(df.head(5))

In [ ]:
# Create binary target: 1 if testscr >= median, 0 otherwise
median_score = df['testscr'].median()
df['high_score'] = (df['testscr'] >= median_score).astype(int)
print(f'Median test score: {median_score:.2f}')
print(df['high_score'].value_counts())

In [ ]:
feature_cols = ['str', 'avginc', 'elpct', 'mealpct', 'expnstu', 'calwpct']
X = df[feature_cols].dropna()
y = df.loc[X.index, 'high_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
lda_acc = accuracy_score(y_test, lda.predict(X_test))
print(f'LDA accuracy: {lda_acc:.4f}')

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test))
print(f'Logistic Regression accuracy: {lr_acc:.4f}')

In [ ]:
# Decision boundary using two features: str (student-teacher ratio) and avginc
feat1, feat2 = 'str', 'avginc'
X2_train = X_train[[feat1, feat2]].values
X2_test  = X_test[[feat1, feat2]].values

lda2 = LinearDiscriminantAnalysis().fit(X2_train, y_train)

x_min, x_max = X[[feat1]].values.min() - 1, X[[feat1]].values.max() + 1
y_min, y_max = X[[feat2]].values.min() - 1, X[[feat2]].values.max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = lda2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
scatter = ax.scatter(X_test[feat1], X_test[feat2],
                     c=y_test, cmap='RdBu', edgecolors='k',
                     linewidths=0.4, alpha=0.8)
ax.set_xlabel(feat1)
ax.set_ylabel(feat2)
ax.set_title('LDA decision boundary\n(two features, test set)')
plt.colorbar(scatter, ax=ax, label='high_score')
plt.tight_layout()
plt.show()

## Your answers here
